In [ ]:
%matplotlib inline

In [ ]:
%matplotlib inline


=============================================================================
EXPERIMENT 2: Accountability & Transparency on OLIVES Ophthalmology Dataset
=============================================================================
FunML Term Project — Georgia Tech ECE 4252/6252

THE SAME BACKWARDS CHAIN, DIFFERENT DOMAIN:

    Phase 1: "Is something wrong overall?"
        → Train a CNN to classify eye disease (DR vs DME)
        → Check: does the model perform equally across patient subgroups?

    Phase 2: "Why did THIS specific diagnosis happen?"
        → Apply Grad-CAM to show WHERE the model looks
        → Test faithfulness: does masking important regions change predictions?
        → Test robustness: do similar images get similar explanations?
        → Compare explanations for CORRECT vs INCORRECT diagnoses

    Phase 3: "Where did this problem come from?"
        → Audit the dataset composition
        → Show that Grad-CAM tells you WHERE but not WHAT
        → Connect to the accountability argument

DATASET: OLIVES (from Prof. AlRegib's lab)
    Task: Classify OCT scans as DR or DME

REQUIREMENTS:
    pip install torch torchvision matplotlib numpy pandas scikit-learn
    pip install grad-cam opencv-python datasets pillow

GPU REQUIRED: Yes (request GPU on PACE before running)
=============================================================================

In [ ]:
import os

olives_path = "/storage/ice1/shared/d-pace_community/makerspace-datasets/MEDICAL/OLIVES/OLIVES"

for root, dirs, files in os.walk(olives_path):
    level = root.replace(olives_path, '').count(os.sep)
    if level < 2:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for f in files[:10]:
            print(f'{subindent}{f}')
        if len(files) > 10:
            print(f'{subindent}... and {len(files) - 10} more files')

In [ ]:
import pandas as pd

csv_path = "/storage/ice1/shared/d-pace_community/makerspace-datasets/MEDICAL/OLIVES/OLIVES/Biomarker_Clinical_Data_Images_Updated.csv"
df = pd.read_csv(csv_path)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nData types:")
print(df.dtypes)

In [ ]:
import os

base = "/storage/ice1/shared/d-pace_community/makerspace-datasets/MEDICAL/OLIVES/OLIVES"

for folder in ['Prime_FULL', 'TREX_DME']:
    path = os.path.join(base, folder)
    contents = os.listdir(path)
    print(f"\n{folder}/  ({len(contents)} items)")
    for item in contents[:5]:
        subpath = os.path.join(path, item)
        if os.path.isdir(subpath):
            subfiles = os.listdir(subpath)
            print(f"  {item}/  ({len(subfiles)} files)")
            for f in subfiles[:3]:
                print(f"    {f}")
        else:
            print(f"  {item}")

In [ ]:
import os
from PIL import Image

base = "/storage/ice1/shared/d-pace_community/makerspace-datasets/MEDICAL/OLIVES/OLIVES"
test_path = df.iloc[0]['Path (Trial/Arm/Folder/Visit/Eye/Image Name)']
full_path = os.path.join(base, test_path)

print(f"Trying: {full_path}")
print(f"Exists: {os.path.exists(full_path)}")

if os.path.exists(full_path):
    img = Image.open(full_path)
    print(f"Image size: {img.size}")
    print(f"Image mode: {img.mode}")
else:
    # Maybe images are nested differently - search for the filename
    filename = os.path.basename(test_path)
    print(f"\nSearching for {filename}...")
    for root, dirs, files in os.walk(base):
        if filename in files:
            found = os.path.join(root, filename)
            print(f"Found at: {found}")
            img = Image.open(found)
            print(f"Image size: {img.size}")
            break

In [ ]:
# Cell 1
%matplotlib inline

# Cell 2 
!pip install grad-cam

# Cell 3 — copy the experiment code into cells and run

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import os
import warnings
warnings.filterwarnings('ignore')
 
# ============================================================================
# CONFIGURATION
# ============================================================================
OLIVES_ROOT = "/storage/ice1/shared/d-pace_community/makerspace-datasets/MEDICAL/OLIVES/OLIVES"
CSV_PATH = os.path.join(OLIVES_ROOT, "Biomarker_Clinical_Data_Images_Updated.csv")
 
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
 
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
 
 
# ============================================================================
# SECTION 1: LOAD & EXPLORE DATA
# ============================================================================
print("\n" + "=" * 70)
print("SECTION 1: Loading OLIVES Dataset")
print("=" * 70)
 
df = pd.read_csv(CSV_PATH)
path_col = 'Path (Trial/Arm/Folder/Visit/Eye/Image Name)'
 
# Extract disease label from path
# TREX_DME/... → DME (label=1)
# Prime_FULL/... → DR  (label=0)
df['disease'] = df[path_col].apply(lambda p: 'DME' if p.startswith('TREX_DME') else 'DR')
df['disease_label'] = df['disease'].apply(lambda d: 0 if d == 'DR' else 1)
 
# Extract patient and eye info
df['trial'] = df[path_col].apply(lambda p: p.split('/')[0])
 
print(f"Total samples: {len(df)}")
print(f"\nDisease distribution:")
print(df['disease'].value_counts())
print(f"\nTrial distribution:")
print(df['trial'].value_counts())
print(f"\nNumber of unique patients: {df['Patient_ID'].nunique()}")
print(f"Number of unique eyes: {df['Eye_ID'].nunique()}")
 
# Biomarker columns
biomarker_cols = [
    'Atrophy / thinning of retinal layers', 'Disruption of EZ', 'DRIL',
    'IR hemorrhages', 'IR HRF', 'Partially attached vitreous face',
    'Fully attached vitreous face', 'Preretinal tissue/hemorrhage',
    'Vitreous debris', 'VMT', 'DRT/ME', 'Fluid (IRF)', 'Fluid (SRF)',
    'Disruption of RPE', 'PED (serous)', 'SHRM'
]
 
print(f"\nBiomarker prevalence:")
for col in biomarker_cols:
    if col in df.columns:
        prev = df[col].mean()
        print(f"  {col:45s}  {prev:.3f}")
 
 
# ============================================================================
# SECTION 2: DATASET CLASS
# ============================================================================
print("\n" + "=" * 70)
print("SECTION 2: Preparing Data")
print("=" * 70)
 
class OLIVESDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.path_col = 'Path (Trial/Arm/Folder/Visit/Eye/Image Name)'
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row[self.path_col])
        
        # Load grayscale image and convert to RGB (ResNet expects 3 channels)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        label = int(row['disease_label'])
        return image, label
 
    def get_metadata(self, idx):
        return self.df.iloc[idx]
 
 
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
 
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
 
# IMPORTANT: Split by PATIENT, not by image
# Otherwise the same patient's scans appear in both train and test → data leakage
patients = df['Patient_ID'].unique()
np.random.shuffle(patients)
split_idx = int(0.8 * len(patients))
train_patients = set(patients[:split_idx])
test_patients = set(patients[split_idx:])
 
df_train = df[df['Patient_ID'].isin(train_patients)]
df_test = df[df['Patient_ID'].isin(test_patients)]
 
train_dataset = OLIVESDataset(df_train, OLIVES_ROOT, transform=train_transform)
test_dataset = OLIVESDataset(df_test, OLIVES_ROOT, transform=test_transform)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
 
print(f"Train patients: {len(train_patients)} | Train images: {len(train_dataset)}")
print(f"Test patients:  {len(test_patients)}  | Test images:  {len(test_dataset)}")
print(f"\nTrain disease distribution:")
print(df_train['disease'].value_counts())
print(f"\nTest disease distribution:")
print(df_test['disease'].value_counts())
 
# Sanity check — load one image
sample_img, sample_label = train_dataset[0]
print(f"\nSample image shape: {sample_img.shape}")
print(f"Sample label: {sample_label} ({'DR' if sample_label == 0 else 'DME'})")
 
 
# ============================================================================
# SECTION 3: TRAIN ResNet-18
# ============================================================================
print("\n" + "=" * 70)
print("SECTION 3: Training ResNet-18")
print("=" * 70)
 
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)  # DR vs DME
model = model.to(DEVICE)
 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
 
train_losses = []
train_accs = []
 
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
 
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
 
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    scheduler.step()
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
 
print("Training complete.")
 
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(train_losses, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[1].plot(train_accs, marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
plt.tight_layout()
plt.show()
 


  PHASE 1: "IS SOMETHING WRONG OVERALL?"

############################################################################

In [ ]:
print("\n" + "=" * 70)
print("PHASE 1: IS SOMETHING WRONG OVERALL?")
print("=" * 70)
 
model.eval()
all_preds = []
all_labels = []
all_probs = []
 
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
 
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
 
disease_names = ['DR', 'DME']
 
print("\n--- Overall Test Performance ---")
print(classification_report(all_labels, all_preds, target_names=disease_names))
 
try:
    auc = roc_auc_score(all_labels, all_probs[:, 1])
    print(f"AUC-ROC: {auc:.4f}")
except Exception as e:
    print(f"AUC-ROC: could not compute ({e})")
 
# Per-class breakdown
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(f"  Predicted →   DR    DME")
print(f"  Actual DR:   {cm[0,0]:4d}  {cm[0,1]:4d}")
if cm.shape[0] > 1:
    print(f"  Actual DME:  {cm[1,0]:4d}  {cm[1,1]:4d}")
 
for i, name in enumerate(disease_names):
    if i < cm.shape[0]:
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        print(f"\n  {name}: TPR = {tpr:.4f} ({tp}/{tp+fn} correctly identified)")
 
# Confidence: right vs wrong
correct_mask = all_preds == all_labels
wrong_mask = all_preds != all_labels
 
correct_conf = all_probs[correct_mask].max(axis=1).mean() if correct_mask.sum() > 0 else 0
wrong_conf = all_probs[wrong_mask].max(axis=1).mean() if wrong_mask.sum() > 0 else 0
 
print(f"\n--- Confidence Analysis ---")
print(f"  When CORRECT: mean confidence = {correct_conf:.4f}")
print(f"  When WRONG:   mean confidence = {wrong_conf:.4f}")
if wrong_conf > 0:
    gap = correct_conf - wrong_conf
    print(f"  Gap: {gap:.4f}")
    if gap < 0.1:
        print(f"  ← MODEL CANNOT SIGNAL WHEN IT'S WRONG")
        print(f"     A doctor cannot rely on confidence to double-check")
 
# Phase 1 figures
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(len(disease_names)))
axes[0].set_yticks(range(len(disease_names)))
axes[0].set_xticklabels(disease_names)
axes[0].set_yticklabels(disease_names)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Phase 1: Confusion Matrix')
for ii in range(cm.shape[0]):
    for jj in range(cm.shape[1]):
        axes[0].text(jj, ii, str(cm[ii, jj]), ha='center', va='center', fontsize=16)
plt.colorbar(im, ax=axes[0])
 
correct_probs_plot = all_probs[correct_mask].max(axis=1)
wrong_probs_plot = all_probs[wrong_mask].max(axis=1)
axes[1].hist(correct_probs_plot, bins=20, alpha=0.7, label='Correct', color='#4CAF50')
if len(wrong_probs_plot) > 0:
    axes[1].hist(wrong_probs_plot, bins=20, alpha=0.7, label='Wrong', color='#F44336')
axes[1].set_xlabel('Model Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Phase 1: Confidence When Right vs Wrong')
axes[1].legend()
plt.tight_layout()
plt.show()
 


  PHASE 2: "WHY DID THIS DIAGNOSIS HAPPEN?" (Grad-CAM)

############################################################################

In [ ]:
print("\n" + "=" * 70)
print("PHASE 2: WHY DID THIS DIAGNOSIS HAPPEN? (Grad-CAM)")
print("=" * 70)
 
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
 
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)
 
 
def get_image_for_display(dataset, idx):
    """Get tensor for model and unnormalized image for display."""
    img_tensor, label = dataset[idx]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_display = (img_tensor * std + mean).permute(1, 2, 0).numpy()
    img_display = np.clip(img_display, 0, 1)
    return img_display, img_tensor, label
 
 
# --- Step 2A: Grad-CAM Visualizations ---
print("\n--- Step 2A: Where Does the Model Look? ---")
 
n_examples = 8
fig, axes = plt.subplots(3, n_examples, figsize=(4 * n_examples, 12))
 
for i in range(n_examples):
    try:
        img_display, img_tensor, true_label = get_image_for_display(test_dataset, i)
        input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
 
        with torch.no_grad():
            output = model(input_tensor)
            pred_label = output.argmax(dim=1).item()
            confidence = torch.softmax(output, dim=1).max().item()
 
        targets = [ClassifierOutputTarget(pred_label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
        cam_overlay = show_cam_on_image(img_display, grayscale_cam, use_rgb=True)
 
        true_name = disease_names[true_label]
        pred_name = disease_names[pred_label]
        mark = '✓' if pred_label == true_label else '✗'
 
        axes[0, i].imshow(img_display)
        axes[0, i].set_title(f'True: {true_name}', fontsize=10)
        axes[0, i].axis('off')
 
        axes[1, i].imshow(cam_overlay)
        axes[1, i].set_title(f'Pred: {pred_name} ({confidence:.2f}) {mark}', fontsize=10)
        axes[1, i].axis('off')
 
        axes[2, i].imshow(grayscale_cam, cmap='jet')
        axes[2, i].set_title('Heatmap', fontsize=10)
        axes[2, i].axis('off')
 
    except Exception as e:
        print(f"  Image {i} error: {e}")
 
axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=12)
axes[2, 0].set_ylabel('Heatmap', fontsize=12)
plt.suptitle('Phase 2: Where Does the Model Look When Diagnosing?', fontsize=14)
plt.tight_layout()
plt.show()
 
 
# --- Step 2B: Faithfulness Test ---
print("\n--- Step 2B: Is Grad-CAM Faithful? ---")
 
def gradcam_faithfulness(model, cam, dataset, n_tests=50):
    top_drops = []
    bottom_drops = []
    model.eval()
 
    for i in range(min(n_tests, len(dataset))):
        try:
            _, img_tensor, _ = get_image_for_display(dataset, i)
            input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
 
            with torch.no_grad():
                pred_class = model(input_tensor).argmax(dim=1).item()
                orig_conf = torch.softmax(model(input_tensor), dim=1)[0, pred_class].item()
 
            targets = [ClassifierOutputTarget(pred_class)]
            heatmap = cam(input_tensor=input_tensor, targets=targets)[0, :]
 
            # Mask top 20% important regions
            top_thresh = np.percentile(heatmap, 80)
            top_mask = torch.tensor(heatmap >= top_thresh).unsqueeze(0).expand(3, -1, -1)
            masked_top = img_tensor.clone()
            masked_top[top_mask] = 0.0
            with torch.no_grad():
                top_conf = torch.softmax(model(masked_top.unsqueeze(0).to(DEVICE)), dim=1)[0, pred_class].item()
            top_drops.append(orig_conf - top_conf)
 
            # Mask bottom 20% (control)
            bot_thresh = np.percentile(heatmap, 20)
            bot_mask = torch.tensor(heatmap <= bot_thresh).unsqueeze(0).expand(3, -1, -1)
            masked_bot = img_tensor.clone()
            masked_bot[bot_mask] = 0.0
            with torch.no_grad():
                bot_conf = torch.softmax(model(masked_bot.unsqueeze(0).to(DEVICE)), dim=1)[0, pred_class].item()
            bottom_drops.append(orig_conf - bot_conf)
        except:
            continue
 
    return top_drops, bottom_drops
 
top_drops, bottom_drops = gradcam_faithfulness(model, cam, test_dataset, n_tests=50)
 
print(f"  Masking TOP 20%:    mean confidence drop = {np.mean(top_drops):.4f}")
print(f"  Masking BOTTOM 20%: mean confidence drop = {np.mean(bottom_drops):.4f}")
ratio = np.mean(top_drops) / max(np.mean(bottom_drops), 0.001)
print(f"  Faithfulness ratio: {ratio:.1f}x")
 
if ratio > 1.5:
    print(f"\n  ✓ Grad-CAM IS faithful — important regions matter more.")
    print(f"    But: faithful ≠ clinically correct. The model might focus on")
    print(f"    the right AREA but use device artifacts, not disease features.")
else:
    print(f"\n  ⚠ Grad-CAM is NOT faithful — highlighted regions don't matter much.")
 
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([top_drops, bottom_drops],
           labels=['Top 20% masked\n(important)', 'Bottom 20% masked\n(unimportant)'])
ax.set_ylabel('Confidence drop')
ax.set_title('Phase 2: Is Grad-CAM Faithful?')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
 
 
# --- Step 2C: Robustness Test ---
print("\n--- Step 2C: Is Grad-CAM Robust? ---")
 
def gradcam_robustness(model, cam, dataset, n_tests=30, noise_std=0.02):
    correlations = []
    for i in range(min(n_tests, len(dataset))):
        try:
            _, img_tensor, _ = get_image_for_display(dataset, i)
            input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
 
            with torch.no_grad():
                pred = model(input_tensor).argmax(dim=1).item()
            targets = [ClassifierOutputTarget(pred)]
            cam_orig = cam(input_tensor=input_tensor, targets=targets)[0, :]
 
            noise = torch.randn_like(img_tensor) * noise_std
            noisy = (img_tensor + noise).unsqueeze(0).to(DEVICE)
            cam_noisy = cam(input_tensor=noisy, targets=targets)[0, :]
 
            corr = np.corrcoef(cam_orig.flatten(), cam_noisy.flatten())[0, 1]
            correlations.append(corr)
        except:
            continue
    return correlations
 
correlations = gradcam_robustness(model, cam, test_dataset, n_tests=30)
 
print(f"  Mean heatmap correlation: {np.mean(correlations):.4f}")
print(f"  Min correlation: {np.min(correlations):.4f}")
print(f"  Cases with correlation < 0.5: {(np.array(correlations) < 0.5).mean():.0%}")
 
if np.mean(correlations) > 0.8:
    print(f"\n  ✓ Explanations are reasonably stable.")
else:
    print(f"\n  ⚠ Explanations are FRAGILE — invisible noise changes the heatmap.")
 
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(correlations, bins=15, edgecolor='black', color='#FF9800', alpha=0.7)
ax.set_xlabel('Heatmap correlation (original vs noisy)')
ax.set_ylabel('Count')
ax.set_title('Phase 2: Grad-CAM Robustness Under Noise')
ax.axvline(x=0.5, color='red', linestyle='--', label='Low robustness threshold')
ax.legend()
plt.tight_layout()
plt.show()
 
 
# --- Step 2D: Correct vs Wrong Explanations ---
print("\n--- Step 2D: Do Correct and Wrong Predictions Look Different? ---")
 
correct_confs_2d = []
wrong_confs_2d = []
count = 0
 
for images, labels in test_loader:
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    with torch.no_grad():
        outputs = model(images)
        confs = torch.softmax(outputs, dim=1)
        preds_batch = outputs.argmax(dim=1)
    for j in range(images.size(0)):
        if count >= 100:
            break
        conf = confs[j, preds_batch[j].item()].item()
        if preds_batch[j].item() == labels[j].item():
            correct_confs_2d.append(conf)
        else:
            wrong_confs_2d.append(conf)
        count += 1
    if count >= 100:
        break
 
print(f"  Correct: mean confidence = {np.mean(correct_confs_2d):.4f} ({len(correct_confs_2d)} cases)")
if len(wrong_confs_2d) > 0:
    print(f"  Wrong:   mean confidence = {np.mean(wrong_confs_2d):.4f} ({len(wrong_confs_2d)} cases)")
    gap = np.mean(correct_confs_2d) - np.mean(wrong_confs_2d)
    print(f"  Gap: {gap:.4f}")
    if gap < 0.1:
        print("""
  ⚠ CRITICAL: The model is equally confident when right and wrong.
     A doctor looking at a Grad-CAM heatmap CANNOT tell whether
     to trust the diagnosis. The explanation looks equally convincing
     for correct and incorrect predictions.
     → This is a FALSE SENSE OF ACCOUNTABILITY.
""")
else:
    print("  No errors in sample — try more samples or harder task.")
 


  PHASE 3: "WHERE DID THIS COME FROM?"

############################################################################

In [ ]:
print("\n" + "=" * 70)
print("PHASE 3: WHERE DID THIS COME FROM?")
print("=" * 70)
 
# Dataset composition
print("\n--- Dataset Governance Audit ---")
print(f"\nPatients by trial:")
print(df.groupby('trial')['Patient_ID'].nunique())
print(f"\nImages by trial:")
print(df['trial'].value_counts())
 
print(f"\nTotal unique patients: {df['Patient_ID'].nunique()}")
print(f"Total unique eyes: {df['Eye_ID'].nunique()}")
print(f"Images per patient (mean): {len(df) / df['Patient_ID'].nunique():.0f}")
 
# BCVA and CST distribution by disease
print(f"\n--- Clinical Label Distribution by Disease ---")
for clinical in ['BCVA', 'CST']:
    print(f"\n  {clinical}:")
    for disease in ['DR', 'DME']:
        vals = df[df['disease'] == disease][clinical].dropna()
        print(f"    {disease}: mean={vals.mean():.1f}, std={vals.std():.1f}, "
              f"min={vals.min():.0f}, max={vals.max():.0f}")
 
print("""
GOVERNANCE CONCERNS:
 
1. POPULATION BIAS:
   Patients come from clinical trials at ONE clinic (Retina Consultants
   of Texas). This is NOT representative of the global diabetic patient
   population. Disease presentation varies across ethnicities, ages,
   and geographies. No demographic information (race, sex, age) is
   provided — so you CANNOT even audit for demographic bias.
   This is itself an accountability failure: if you can't measure
   bias, you can't detect it.
 
2. TRIAL SELECTION BIAS:
   Patients in clinical trials are pre-screened. They're typically
   healthier, more compliant, and more accessible than the general
   patient population. A model trained on trial patients may fail
   on typical clinic patients who are sicker, less monitored, or
   have more complex conditions.
 
3. LABELING SUBJECTIVITY:
   The 16 biomarker labels are human-interpreted from OCT scans.
   Different graders can disagree on biomarker presence. With limited
   graders, one person's interpretation becomes ground truth. The
   model learns their biases and blind spots.
 
4. DEVICE DEPENDENCY:
   All OCT scans come from the same device type. Different manufacturers
   produce scans with different contrast, resolution, and noise patterns.
   The model may learn device-specific artifacts rather than disease
   features. Grad-CAM might highlight the "right area" while the model
   actually relies on texture patterns specific to this one device.
 
5. NO DEMOGRAPHIC DATA:
   Unlike UCI Adult where we could measure bias by race and sex,
   OLIVES provides NO patient demographics. This means:
   - We cannot check if the model performs differently across
     racial or ethnic groups
   - We cannot audit for health equity
   - We cannot know if certain populations are underrepresented
   This is the data governance equivalent of blindfolding yourself
   and claiming you don't see any problems.
""")
 
# Phase 3 figures
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
# Class distribution
dr_total = (df['disease'] == 'DR').sum()
dme_total = (df['disease'] == 'DME').sum()
axes[0].bar(['DR', 'DME'], [dr_total, dme_total], color=['#2196F3', '#FF9800'], edgecolor='black')
axes[0].set_title('Phase 3: Class Distribution')
axes[0].set_ylabel('Number of OCT scans')
 
# Patients per trial
trial_counts = df.groupby('trial')['Patient_ID'].nunique()
trial_counts.plot(kind='bar', ax=axes[1], color='#4CAF50', edgecolor='black')
axes[1].set_title('Phase 3: Patients by Trial')
axes[1].set_ylabel('Unique patients')
axes[1].tick_params(axis='x', rotation=0)
 
# BCVA by disease
dr_bcva = df[df['disease'] == 'DR']['BCVA'].dropna()
dme_bcva = df[df['disease'] == 'DME']['BCVA'].dropna()
axes[2].hist(dr_bcva, bins=20, alpha=0.7, label='DR', color='#2196F3')
axes[2].hist(dme_bcva, bins=20, alpha=0.7, label='DME', color='#FF9800')
axes[2].set_xlabel('BCVA (Visual Acuity)')
axes[2].set_ylabel('Count')
axes[2].set_title('Phase 3: Visual Acuity by Disease')
axes[2].legend()
 
plt.tight_layout()
plt.show()
 
 
# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 70)
print("EXPERIMENT 2 COMPLETE")
print("=" * 70)
print("""
CONNECTING BOTH EXPERIMENTS:
 
  Experiment 1 (UCI Adult — lending):
    → SHAP hides sex discrimination behind proxy features
    → The explanation says "relationship status"
    → The reality is "you're a woman"
    → Explainability MASKS bias
 
  Experiment 2 (OLIVES — healthcare):
    → Grad-CAM shows WHERE the model looks but not WHAT it sees
    → The heatmap highlights the right retinal area
    → But the model might rely on device artifacts, not disease
    → When the model is WRONG, the heatmap looks equally convincing
    → AND we can't even check for demographic bias because the
       dataset provides no demographic data
    → Explainability creates FALSE CONFIDENCE
 
  SAME CONCLUSION:
    Explainability tools are faithful to the model.
    But faithful to a biased/unreliable model ≠ accountability.
 
    Accountability requires:
    - Mandatory demographic data collection for bias auditing
    - Multi-site validation before deployment
    - Uncertainty-based routing to human specialists
    - Contestability rights for patients
    - Independent third-party auditing
    - Data governance standards (who is represented, who is missing)
""")
 

In [ ]:
# ============================================================
# EXPERIMENT 2 DEEP DIVE: ACCOUNTABILITY TESTS
# ============================================================

# TEST 1: Is the model learning disease or trial artifacts?
print("=" * 60)
print("TEST 1: DISEASE DETECTION OR TRIAL DETECTION?")
print("=" * 60)

# Check: do correct and incorrect predictions cluster by patient?
test_df = df_test.reset_index(drop=True)
test_preds_series = pd.Series(all_preds)

test_df['predicted'] = all_preds
test_df['correct'] = (all_preds == all_labels)

# Per-patient accuracy
patient_acc = test_df.groupby('Patient_ID').agg(
    n_images=('correct', 'count'),
    accuracy=('correct', 'mean'),
    disease=('disease', 'first')
).sort_values('accuracy')

print("\nPer-patient accuracy (worst patients first):")
print(patient_acc.to_string())

# Are errors concentrated in specific patients?
print(f"\nPatients with <50% accuracy: {(patient_acc['accuracy'] < 0.5).sum()}")
print(f"Patients with 100% accuracy: {(patient_acc['accuracy'] == 1.0).sum()}")
print(f"Total test patients: {len(patient_acc)}")

print("""
INTERPRETATION:
  If some patients have 0% accuracy and others have 100%, the model
  memorized patient-specific patterns, not disease patterns. A new
  patient from a different clinic would get unpredictable results.
  
  This is an accountability failure: the model LOOKS like it detects
  disease (81% overall) but may actually detect "which trial/device
  produced this scan."
""")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3' if d == 'DR' else '#FF9800' for d in patient_acc['disease']]
ax.bar(range(len(patient_acc)), patient_acc['accuracy'], color=colors)
ax.set_xlabel('Patient (sorted by accuracy)')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Patient Accuracy: Does the Model Work for Everyone?')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random chance')
# Custom legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2196F3', label='DR'),
                   Patch(facecolor='#FF9800', label='DME'),
                   plt.Line2D([0], [0], color='red', linestyle='--', label='Random chance')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

In [ ]:
# TEST 2: Does Grad-CAM focus on biomarker regions?
print("\n" + "=" * 60)
print("TEST 2: DOES GRAD-CAM CORRELATE WITH ACTUAL BIOMARKERS?")
print("=" * 60)

# For each test image, compare:
# - Does the model predict correctly?
# - Which biomarkers are actually present?
# - Does Grad-CAM focus (spread) relate to biomarker count?

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

biomarker_cols = [
    'Atrophy / thinning of retinal layers', 'Disruption of EZ', 'DRIL',
    'IR hemorrhages', 'IR HRF', 'Partially attached vitreous face',
    'Fully attached vitreous face', 'Preretinal tissue/hemorrhage',
    'Vitreous debris', 'VMT', 'DRT/ME', 'Fluid (IRF)', 'Fluid (SRF)',
    'Disruption of RPE', 'PED (serous)', 'SHRM'
]

focus_scores = []
biomarker_counts = []
correctness = []

n_analyze = min(100, len(test_dataset))

for i in range(n_analyze):
    try:
        img_display, img_tensor, label = get_image_for_display(test_dataset, i)
        input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            output = model(input_tensor)
            pred = output.argmax(dim=1).item()
            conf = torch.softmax(output, dim=1).max().item()
        
        targets = [ClassifierOutputTarget(pred)]
        heatmap = cam(input_tensor=input_tensor, targets=targets)[0, :]
        
        # Measure heatmap focus (higher = more concentrated)
        heatmap_norm = heatmap / (heatmap.sum() + 1e-8)
        focus = (heatmap_norm ** 2).sum()
        
        # Count biomarkers present for this image
        row = test_df.iloc[i]
        n_biomarkers = sum(row[col] == 1 for col in biomarker_cols if col in row.index and pd.notna(row[col]))
        
        focus_scores.append(focus)
        biomarker_counts.append(n_biomarkers)
        correctness.append(pred == label)
    except:
        continue

focus_scores = np.array(focus_scores)
biomarker_counts = np.array(biomarker_counts)
correctness = np.array(correctness)

# Correlation between focus and biomarkers
corr = np.corrcoef(focus_scores, biomarker_counts)[0, 1]
print(f"\nCorrelation between Grad-CAM focus and biomarker count: {corr:.4f}")

if abs(corr) < 0.2:
    print("""
  ← WEAK CORRELATION: Grad-CAM attention does NOT relate to
     the number of clinical biomarkers present. The model focuses
     on regions for reasons UNRELATED to known disease markers.
     
     A doctor seeing the heatmap would assume the model found
     clinically relevant features. It likely didn't.
""")
else:
    print(f"  Moderate/strong correlation — model may be using biomarker-related features.")

# Does biomarker count affect accuracy?
print("Accuracy by biomarker count:")
for n in sorted(set(biomarker_counts)):
    mask = biomarker_counts == n
    if mask.sum() >= 5:
        acc = correctness[mask].mean()
        print(f"  {n} biomarkers present: accuracy = {acc:.2%} (n={mask.sum()})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(biomarker_counts, focus_scores, alpha=0.5, 
                c=['green' if c else 'red' for c in correctness])
axes[0].set_xlabel('Number of biomarkers present')
axes[0].set_ylabel('Grad-CAM focus score')
axes[0].set_title(f'Does Grad-CAM Relate to Biomarkers?\n(correlation: {corr:.3f})')
from matplotlib.lines import Line2D
legend = [Line2D([0], [0], marker='o', color='w', markerfacecolor='green', label='Correct'),
          Line2D([0], [0], marker='o', color='w', markerfacecolor='red', label='Wrong')]
axes[0].legend(handles=legend)

axes[1].bar(['Correct', 'Wrong'], 
            [focus_scores[correctness].mean(), focus_scores[~correctness].mean() if (~correctness).sum() > 0 else 0],
            color=['#4CAF50', '#F44336'], edgecolor='black')
axes[1].set_ylabel('Mean Grad-CAM focus')
axes[1].set_title('Grad-CAM Focus: Correct vs Wrong Predictions')
plt.tight_layout()
plt.show()

In [ ]:
# TEST 3: Image perturbation robustness (like deepfake compression test)
print("\n" + "=" * 60)
print("TEST 3: DOES THE MEDICAL DETECTOR SURVIVE PERTURBATIONS?")
print("=" * 60)

import torchvision.transforms.functional as TF

perturbations = {
    'Original': lambda img: img,
    'Brightness +20%': lambda img: TF.adjust_brightness(img, 1.2),
    'Brightness -20%': lambda img: TF.adjust_brightness(img, 0.8),
    'Contrast +30%': lambda img: TF.adjust_contrast(img, 1.3),
    'Gaussian noise': lambda img: img + torch.randn_like(img) * 0.05,
    'Blur': lambda img: TF.gaussian_blur(img, kernel_size=5),
}

n_perturb = min(50, len(test_dataset))
perturb_results = {name: {'flipped': 0, 'total': 0} for name in perturbations}

for i in range(n_perturb):
    try:
        _, img_tensor, label = get_image_for_display(test_dataset, i)
        
        # Original prediction
        with torch.no_grad():
            orig_pred = model(img_tensor.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
        
        for name, perturb_fn in perturbations.items():
            perturbed = perturb_fn(img_tensor.clone())
            with torch.no_grad():
                new_pred = model(perturbed.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
            
            perturb_results[name]['total'] += 1
            perturb_results[name]['flipped'] += int(new_pred != orig_pred)
    except:
        continue

print(f"\n{'Perturbation':<25s} {'Predictions Flipped':>20s}")
print("-" * 47)
for name, r in perturb_results.items():
    if r['total'] > 0:
        flip_rate = r['flipped'] / r['total']
        print(f"{name:<25s} {flip_rate:>20.1%}")

print("""
INTERPRETATION:
  If simple brightness or contrast changes flip the diagnosis,
  a doctor using a slightly different monitor or a hospital
  with different lighting in their imaging room would get
  different AI diagnoses for the same patient.
  
  This is an accountability nightmare: the diagnosis depends
  on imaging conditions, not just the patient's disease.
""")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
names = [n for n in perturbations.keys() if n != 'Original']
rates = [perturb_results[n]['flipped'] / max(perturb_results[n]['total'], 1) for n in names]
colors = ['#F44336' if r > 0.1 else '#FF9800' if r > 0.05 else '#4CAF50' for r in rates]
ax.bar(names, rates, color=colors, edgecolor='black')
ax.set_ylabel('Prediction Flip Rate')
ax.set_title('Medical AI Robustness: Do Small Changes Flip Diagnoses?')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# TEST 4: What the model CANNOT see (no demographics)
print("\n" + "=" * 60)
print("TEST 4: THE INVISIBLE BIAS PROBLEM")
print("=" * 60)

print(f"""
OLIVES dataset columns: {list(df.columns)}

Columns that relate to patient identity:
  - Patient_ID: {df['Patient_ID'].nunique()} unique patients
  - Eye_ID: {df['Eye_ID'].nunique()} unique eyes

Columns that DON'T EXIST:
  - Age           ← missing
  - Sex           ← missing  
  - Race/ethnicity ← missing
  - Socioeconomic status ← missing
  - Geographic location ← missing (beyond "Texas")

COMPARISON WITH EXPERIMENT 1 (UCI Adult):

  UCI Adult: 14 features, INCLUDING race and sex
  → We DETECTED a disparate impact ratio of 0.32
  → We PROVED discrimination through proxy features
  → We MEASURED the bias and tested fixes

  OLIVES: 22 columns, ZERO demographics
  → We CANNOT detect demographic disparities
  → We CANNOT check if the model fails for certain populations
  → We CANNOT audit for health equity
  → We don't even know if the 87 patients represent the
     population that will USE this system

  In Experiment 1, the bias was visible and measurable.
  In Experiment 2, the bias — if it exists — is INVISIBLE.
  
  Which is more dangerous?
  
  THE ACCOUNTABILITY PARADOX:
  A system you CAN audit (Experiment 1) looks worse because
  you can see its flaws. A system you CANNOT audit (Experiment 2)
  looks clean because the flaws are hidden.
  
  This creates a perverse incentive: DON'T collect demographic
  data, and your system will APPEAR fairer because nobody can
  prove it isn't.
  
  Real accountability requires MANDATING demographic data
  collection specifically so that bias CAN be measured.
""")

=============================================================================
EXPERIMENT 2: ALL FIGURES AND TABLES FOR REPORT
=============================================================================
Run this AFTER running experiment_2_olives.py so all variables exist.
Saves all figures as numbered PNGs and prints all tables.
=============================================================================

In [ ]:
import os
SAVE_DIR = "./report_figures_exp2"
os.makedirs(SAVE_DIR, exist_ok=True)

%matplotlib inline
from sklearn.metrics import accuracy_score, classification_report

# ============================================================
# FIGURE 1: Training Curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(train_losses, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[1].plot(train_accs, marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig01_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 2: Confusion Matrix + Confidence Distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(len(disease_names)))
axes[0].set_yticks(range(len(disease_names)))
axes[0].set_xticklabels(['DR', 'DME'])
axes[0].set_yticklabels(['DR', 'DME'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for ii in range(cm.shape[0]):
    for jj in range(cm.shape[1]):
        axes[0].text(jj, ii, str(cm[ii, jj]), ha='center', va='center', fontsize=16)
plt.colorbar(im, ax=axes[0])

correct_mask = all_preds == all_labels
wrong_mask = all_preds != all_labels
correct_probs_plot = all_probs[correct_mask].max(axis=1)
wrong_probs_plot = all_probs[wrong_mask].max(axis=1)
axes[1].hist(correct_probs_plot, bins=20, alpha=0.7, label='Correct', color='#4CAF50')
if len(wrong_probs_plot) > 0:
    axes[1].hist(wrong_probs_plot, bins=20, alpha=0.7, label='Wrong', color='#F44336')
axes[1].set_xlabel('Model Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence: Right vs Wrong')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig02_confusion_confidence.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 3: Grad-CAM Examples
# ============================================================
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

n_examples = 8
fig, axes = plt.subplots(3, n_examples, figsize=(4 * n_examples, 12))

for i in range(n_examples):
    try:
        img_display, img_tensor, true_label = get_image_for_display(test_dataset, i)
        input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output = model(input_tensor)
            pred_label = output.argmax(dim=1).item()
            confidence = torch.softmax(output, dim=1).max().item()
        targets = [ClassifierOutputTarget(pred_label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
        cam_overlay = show_cam_on_image(img_display, grayscale_cam, use_rgb=True)

        true_name = 'DR' if true_label == 0 else 'DME'
        pred_name = 'DR' if pred_label == 0 else 'DME'
        mark = '✓' if pred_label == true_label else '✗'

        axes[0, i].imshow(img_display)
        axes[0, i].set_title(f'True: {true_name}', fontsize=10)
        axes[0, i].axis('off')
        axes[1, i].imshow(cam_overlay)
        axes[1, i].set_title(f'Pred: {pred_name} ({confidence:.2f}) {mark}', fontsize=10)
        axes[1, i].axis('off')
        axes[2, i].imshow(grayscale_cam, cmap='jet')
        axes[2, i].set_title('Heatmap', fontsize=10)
        axes[2, i].axis('off')
    except Exception as e:
        print(f"  Image {i} error: {e}")

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=12)
axes[2, 0].set_ylabel('Heatmap', fontsize=12)
plt.suptitle('Where Does the Model Look When Diagnosing?', fontsize=14)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig03_gradcam_examples.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 4: Grad-CAM Faithfulness
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([top_drops, bottom_drops],
           labels=['Top 20% masked\n(important)', 'Bottom 20% masked\n(unimportant)'])
ax.set_ylabel('Confidence drop')
ax.set_title('Grad-CAM Faithfulness Test')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig04_gradcam_faithfulness.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 5: Grad-CAM Robustness
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(correlations, bins=15, edgecolor='black', color='#FF9800', alpha=0.7)
ax.set_xlabel('Heatmap correlation (original vs noisy)')
ax.set_ylabel('Count')
ax.set_title('Grad-CAM Robustness Under Noise')
ax.axvline(x=0.5, color='red', linestyle='--', label='Low robustness threshold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig05_gradcam_robustness.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 6: Per-Patient Accuracy
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
colors_pat = ['#2196F3' if d == 'DR' else '#FF9800' for d in patient_acc['disease']]
ax.bar(range(len(patient_acc)), patient_acc['accuracy'], color=colors_pat)
ax.set_xlabel('Patient (sorted by accuracy)')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Patient Accuracy: Does the Model Work for Everyone?')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random chance')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2196F3', label='DR'),
                   Patch(facecolor='#FF9800', label='DME'),
                   plt.Line2D([0], [0], color='red', linestyle='--', label='Random chance')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig06_per_patient_accuracy.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 7: Biomarker Correlation
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(biomarker_counts, focus_scores, alpha=0.5,
                c=['green' if c else 'red' for c in correctness])
axes[0].set_xlabel('Number of biomarkers present')
axes[0].set_ylabel('Grad-CAM focus score')
corr_val = np.corrcoef(focus_scores, biomarker_counts)[0, 1]
axes[0].set_title(f'Grad-CAM vs Biomarkers (r={corr_val:.3f})')
from matplotlib.lines import Line2D
legend = [Line2D([0], [0], marker='o', color='w', markerfacecolor='green', label='Correct'),
          Line2D([0], [0], marker='o', color='w', markerfacecolor='red', label='Wrong')]
axes[0].legend(handles=legend)

axes[1].bar(['Correct', 'Wrong'],
            [focus_scores[correctness].mean(),
             focus_scores[~correctness].mean() if (~correctness).sum() > 0 else 0],
            color=['#4CAF50', '#F44336'], edgecolor='black')
axes[1].set_ylabel('Mean Grad-CAM focus')
axes[1].set_title('Grad-CAM Focus: Correct vs Wrong')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig07_biomarker_correlation.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 8: Perturbation Robustness
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
perturb_names = [n for n in perturb_results.keys() if n != 'Original']
perturb_rates = [perturb_results[n]['flipped'] / max(perturb_results[n]['total'], 1) for n in perturb_names]
colors_p = ['#F44336' if r > 0.1 else '#FF9800' if r > 0.05 else '#4CAF50' for r in perturb_rates]
ax.bar(perturb_names, [r * 100 for r in perturb_rates], color=colors_p, edgecolor='black')
ax.set_ylabel('Predictions Flipped (%)')
ax.set_title('Medical AI Robustness: Do Small Changes Flip Diagnoses?')
for i, v in enumerate(perturb_rates):
    ax.text(i, v * 100 + 1, f'{v:.0%}', ha='center', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig08_perturbation_robustness.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# FIGURE 9: Data Governance (class distribution + BCVA)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dr_total = (df['disease'] == 'DR').sum()
dme_total = (df['disease'] == 'DME').sum()
axes[0].bar(['DR', 'DME'], [dr_total, dme_total], color=['#2196F3', '#FF9800'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of OCT scans')

trial_counts = df.groupby('trial')['Patient_ID'].nunique()
trial_counts.plot(kind='bar', ax=axes[1], color='#4CAF50', edgecolor='black')
axes[1].set_title('Patients by Trial')
axes[1].set_ylabel('Unique patients')
axes[1].tick_params(axis='x', rotation=0)

dr_bcva = df[df['disease'] == 'DR']['BCVA'].dropna()
dme_bcva = df[df['disease'] == 'DME']['BCVA'].dropna()
axes[2].hist(dr_bcva, bins=20, alpha=0.7, label='DR', color='#2196F3')
axes[2].hist(dme_bcva, bins=20, alpha=0.7, label='DME', color='#FF9800')
axes[2].set_xlabel('BCVA (Visual Acuity)')
axes[2].set_ylabel('Count')
axes[2].set_title('Visual Acuity by Disease')
axes[2].legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig09_data_governance.png', dpi=200, bbox_inches='tight')
plt.show()

# ============================================================
# TABLE 1: Overall Test Performance
# ============================================================
print("\n" + "=" * 80)
print("TABLE 1: OVERALL TEST PERFORMANCE")
print("=" * 80)
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=['DR', 'DME']))
try:
    from sklearn.metrics import roc_auc_score
    print(f"AUC-ROC: {roc_auc_score(all_labels, all_probs[:, 1]):.4f}")
except:
    pass

# ============================================================
# TABLE 2: Per-Patient Accuracy
# ============================================================
print("\n" + "=" * 80)
print("TABLE 2: PER-PATIENT ACCURACY")
print("=" * 80)
print(patient_acc.to_string())
print(f"\nPatients with <50% accuracy: {(patient_acc['accuracy'] < 0.5).sum()}")
print(f"Patients with 100% accuracy: {(patient_acc['accuracy'] == 1.0).sum()}")

# ============================================================
# TABLE 3: Faithfulness & Robustness Results
# ============================================================
print("\n" + "=" * 80)
print("TABLE 3: GRAD-CAM FAITHFULNESS & ROBUSTNESS")
print("=" * 80)
print(f"\nFaithfulness:")
print(f"  Masking TOP 20%:    confidence drop = {np.mean(top_drops):.4f}")
print(f"  Masking BOTTOM 20%: confidence drop = {np.mean(bottom_drops):.4f}")
ratio = np.mean(top_drops) / max(np.mean(bottom_drops), 0.001)
print(f"  Faithfulness ratio: {ratio:.1f}x")
print(f"\nRobustness:")
print(f"  Mean heatmap correlation: {np.mean(correlations):.4f}")
print(f"  Min correlation: {np.min(correlations):.4f}")

# ============================================================
# TABLE 4: Perturbation Results
# ============================================================
print("\n" + "=" * 80)
print("TABLE 4: PERTURBATION ROBUSTNESS")
print("=" * 80)
print(f"\n{'Perturbation':<25s} {'Predictions Flipped':>20s}")
print("-" * 47)
for name, r in perturb_results.items():
    if r['total'] > 0:
        flip_rate = r['flipped'] / r['total']
        print(f"{name:<25s} {flip_rate:>20.1%}")

# ============================================================
# TABLE 5: Confidence Analysis
# ============================================================
print("\n" + "=" * 80)
print("TABLE 5: CONFIDENCE ANALYSIS")
print("=" * 80)
correct_conf = all_probs[correct_mask].max(axis=1).mean()
wrong_conf = all_probs[wrong_mask].max(axis=1).mean() if wrong_mask.sum() > 0 else 0
print(f"  When CORRECT: {correct_conf:.4f}")
print(f"  When WRONG:   {wrong_conf:.4f}")
print(f"  Gap:          {correct_conf - wrong_conf:.4f}")

# ============================================================
# TABLE 6: Biomarker Prevalence
# ============================================================
print("\n" + "=" * 80)
print("TABLE 6: BIOMARKER PREVALENCE")
print("=" * 80)
biomarker_cols = [
    'Atrophy / thinning of retinal layers', 'Disruption of EZ', 'DRIL',
    'IR hemorrhages', 'IR HRF', 'Partially attached vitreous face',
    'Fully attached vitreous face', 'Preretinal tissue/hemorrhage',
    'Vitreous debris', 'VMT', 'DRT/ME', 'Fluid (IRF)', 'Fluid (SRF)',
    'Disruption of RPE', 'PED (serous)', 'SHRM'
]
for col in biomarker_cols:
    if col in df.columns:
        print(f"  {col:45s}  {df[col].mean():.3f}")

# ============================================================
# TABLE 7: Data Governance Summary
# ============================================================
print("\n" + "=" * 80)
print("TABLE 7: DATA GOVERNANCE AUDIT")
print("=" * 80)
print(f"  Total samples: {len(df)}")
print(f"  Unique patients: {df['Patient_ID'].nunique()}")
print(f"  Unique eyes: {df['Eye_ID'].nunique()}")
print(f"  Images per patient: {len(df) / df['Patient_ID'].nunique():.0f}")
print(f"  DR samples: {(df['disease']=='DR').sum()} ({(df['disease']=='DR').mean():.1%})")
print(f"  DME samples: {(df['disease']=='DME').sum()} ({(df['disease']=='DME').mean():.1%})")
print(f"  Source: Retina Consultants of Texas (single clinic)")
print(f"  Demographics available: NONE (no race, sex, age)")
print(f"  Train accuracy: {train_accs[-1]:.4f}")
print(f"  Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"  Gap (overfitting): {train_accs[-1] - accuracy_score(all_labels, all_preds):.4f}")

# ============================================================
# DONE
# ============================================================
print("\n" + "=" * 80)
print(f"ALL FIGURES SAVED TO: {SAVE_DIR}/")
print("=" * 80)
for f in sorted(os.listdir(SAVE_DIR)):
    print(f"  {f}")